# 24 · Pre-test hyperparameter search with separate run identities

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Optional development search. Never reuse a completed run directory for a changed learning rate.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Select a small prespecified validation search

In [ ]:
from dataclasses import replace
from oncoplate.pipeline import load_study,spec_from_cfg
from oncoplate.training import train_predictor
records,targets=load_study(cfg,'joint',stage_images=True)
BACKBONE='resnet50';REGIME='frozen'
base=spec_from_cfg(cfg,BACKBONE,REGIME,'joint',0)
LEARNING_RATES=[.0001,.001] if REGIME=='frozen' else [.00001,.00003]

## 2. Fit candidates in distinct directories

In [ ]:
rows=[]
for learning_rate in LEARNING_RATES:
    spec=replace(base,lr=learning_rate)
    out=p['runs']/'development_hyperparameters'/spec.run_id/f'lr_{learning_rate:g}'
    train_predictor(records,targets,spec,out,p['features'])
    completed=read_json(out/'complete.json')
    rows.append({'backbone':BACKBONE,'regime':REGIME,'learning_rate':learning_rate,'validation_bce':completed['best_validation_loss'],'run_dir':str(out)})
import pandas as pd
search=pd.DataFrame(rows);write_table(p['reports']/'hyperparameter_validation_search.csv',search);display(search)

## 3. Freeze the selected settings before the main grid

In [ ]:
best=search.sort_values(['validation_bce','learning_rate']).iloc[0].to_dict()
write_json(p['private']/'proposed_hyperparameter_choice.json',best)
print(best)
print('Update the reviewed configuration and record the amendment before starting the principal 60-run grid. No test values are consulted.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
